In [9]:
import cv2 as cv
import numpy as np
import math
import os
import random
from glob import glob

# --- Configuration ---
SYMBOLS_DIR = "data/raw/templates" # Contains folders: chain, double, etc.
OUTPUT_IMG_DIR = "data/generated/images"
OUTPUT_LBL_DIR = "data/generated/labels"
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_LBL_DIR, exist_ok=True)

# 1. Update CLASS_MAP to include everything in your folders
CLASS_MAP = {
    "chain": 0, 
    "single": 1, 
    "double": 2, 
    "half_double": 3,
    "treble": 4,
    "fan": 5
}

# 2. Define probabilities for motifs
MOTIF_SYMBOLS = ["chain", "double", "half_double", "treble", "single", "fan"]
MOTIF_WEIGHTS = [0.4, 0.1, 0.1, 0.1, 0.1, 0.2]
FAN_SYMBOLS = ["double", "half_double", "treble"]

def get_random_template(symbol_name):
    """Picks a random image file from the symbol's specific folder."""
    folder_path = os.path.join(SYMBOLS_DIR, symbol_name)
    # Support both .png and .jpg templates
    files = glob(os.path.join(folder_path, "*.png")) + glob(os.path.join(folder_path, "*.jpg"))
    
    if not files:
        return None
    
    selected_file = random.choice(files)
    img = cv.imread(selected_file, cv.IMREAD_UNCHANGED)
    return img

def remove_background(img):
    if img is None: return None
    if img.shape[2] == 4: return img
    bgr = img[:, :, :3]
    gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
    _, mask = cv.threshold(gray, 240, 255, cv.THRESH_BINARY_INV)
    mask = cv.GaussianBlur(mask, (3, 3), 0)
    return cv.merge([bgr[:,:,0], bgr[:,:,1], bgr[:,:,2], mask])

def get_rotated_corners(x_ctr, y_ctr, width, height, angle_deg):
    angle_rad = math.radians(angle_deg)
    cos_a, sin_a = math.cos(angle_rad), math.sin(angle_rad)
    w2, h2 = width / 2, height / 2
    corners = [(-w2, -h2), (w2, -h2), (w2, h2), (-w2, h2)]
    return [(x * cos_a - y * sin_a + x_ctr, x * sin_a + y * cos_a + y_ctr) for x, y in corners]

def overlay_symbol(canvas, symbol, pos, angle, class_name):
    """Pastes symbol without clipping and returns OBB label string."""
    h_c, w_c = canvas.shape[:2]
    sh, sw = symbol.shape[:2]
    curr_x, curr_y = pos
    
    # 1. Calculate the size of the new image that will hold the rotated symbol
    # We need a larger box so the corners don't get "cut"
    angle_rad = math.radians(-angle - 90)
    cos = abs(math.cos(angle_rad))
    sin = abs(math.sin(angle_rad))
    new_sw = int((sh * sin) + (sw * cos))
    new_sh = int((sh * cos) + (sw * sin))

    # 2. Adjust rotation matrix to include translation (to center it in the new box)
    M = cv.getRotationMatrix2D((sw/2, sh/2), -angle - 90, 1.0)
    M[0, 2] += (new_sw / 2) - sw / 2
    M[1, 2] += (new_sh / 2) - sh / 2

    # 3. Rotate with the new calculated size
    rotated_sym = cv.warpAffine(
        symbol, M, (new_sw, new_sh), 
        flags=cv.INTER_LINEAR, 
        borderMode=cv.BORDER_CONSTANT, 
        borderValue=(0,0,0,0)
    )
    
    # 4. Paste logic using the new dimensions
    y1, y2 = int(curr_y - new_sh/2), int(curr_y + new_sh/2)
    x1, x2 = int(curr_x - new_sw/2), int(curr_x + new_sw/2)
    
    # Ensure coordinates are within canvas boundaries
    if y1 >= 0 and y2 < h_c and x1 >= 0 and x2 < w_c:
        # Use the actual dimensions of the slice for the alpha blend
        # This handles small rounding errors in coordinate math
        region = canvas[y1:y2, x1:x2]
        r_h, r_w = region.shape[:2]
        rotated_sym = rotated_sym[:r_h, :r_w] # Crop symbol to match region if needed
        
        alpha_s = rotated_sym[:, :, 3] / 255.0
        alpha_l = 1.0 - alpha_s
        
        for c in range(3):
            canvas[y1:y2, x1:x2, c] = (alpha_s * rotated_sym[:, :, c] + 
                                      alpha_l * canvas[y1:y2, x1:x2, c])
        
        # 5. Generate OBB Label (Uses original sw/sh because we want the box 
        # to fit the symbol, not the padded empty space)
        corners = get_rotated_corners(curr_x, curr_y, sw, sh, angle + 90)
        return f"{CLASS_MAP[class_name]} " + " ".join([f"{cx/w_c:.6f} {cy/h_c:.6f}" for cx, cy in corners])
    
    return None

def generate_fan(canvas, labels):
    pivot_x, pivot_y = random.randint(150, 450), random.randint(300, 500)
    symbol_name = random.choice(FAN_SYMBOLS)
    
    # NEW: Load random template from folder
    raw_img = get_random_template(symbol_name)
    template = remove_background(raw_img)
    if template is None: return
    
    num_symbols = random.randint(3, 7)
    base_angle, spread = -90, random.randint(60, 140)
    
    for i in range(num_symbols):
        angle = base_angle - (spread/2) + (i * (spread/(num_symbols-1)))
        scale = random.uniform(0.7, 1.1)
        sh, sw = int(template.shape[0]*scale), int(template.shape[1]*scale)
        resized = cv.resize(template, (sw, sh))
        dist = sh / 2
        pos = (pivot_x + dist * math.cos(math.radians(angle)), pivot_y + dist * math.sin(math.radians(angle)))
        lbl = overlay_symbol(canvas, resized, pos, angle, symbol_name)
        if lbl: labels.append(lbl)

def generate_motif(canvas, labels):
    start_x, y = random.randint(50, 100), random.randint(100, 500)
    pattern_length = random.randint(6, 12)
    spacing = 55 
    
    for i in range(pattern_length):
        symbol_name = random.choices(MOTIF_SYMBOLS, weights=MOTIF_WEIGHTS, k=1)[0]
        
        # NEW: Load random template from folder
        raw_img = get_random_template(symbol_name)
        template = remove_background(raw_img)
        if template is None: continue
            
        if symbol_name == "chain":
            scale = random.uniform(0.35, 0.55)
        elif symbol_name == "fan":
            scale = random.uniform(1.0, 1.3)
        else:
            scale = random.uniform(0.7, 0.9)
            
        sh, sw = int(template.shape[0]*scale), int(template.shape[1]*scale)
        resized = cv.resize(template, (sw, sh))
        pos = (start_x + (i * spacing), y + random.randint(-8, 8))
        lbl = overlay_symbol(canvas, resized, pos, -90, symbol_name)
        if lbl: labels.append(lbl)

# --- Execution ---
for i in range(200):
    canvas = np.ones((640, 640, 3), dtype=np.uint8) * 255
    labels = []
    
    if random.random() < 0.5:
        generate_fan(canvas, labels)
    else:
        generate_motif(canvas, labels)

    if labels:
        name = f"synth_{i}"
        cv.imwrite(f"{OUTPUT_IMG_DIR}/{name}.jpg", canvas)
        with open(f"{OUTPUT_LBL_DIR}/{name}.txt", "w") as f:
            f.write("\n".join(labels))